<a href="https://colab.research.google.com/github/haneef5244/google-colab/blob/langgraph-reason-act/langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
from google.colab import userdata
import os

# Explicitly set environment variables to ensure they are picked up immediately
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "langgraph-test"
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["TAVILY_API_KEY"]="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"

In [ ]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama pull qwen3.5:9b
!ollama pull dolphin-llama3
!ollama pull richardyoung/qwen2.5-14b-instruct-abliterated

In [ ]:
!pip install langchain langchain-ollama langchain-tavily python-dotenv langgraph

In [ ]:
# react.py
# Holds all reasoning logic that will be used in graph
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_tavily import TavilySearch
from datetime import date

load_dotenv()

@tool
def triple(num: float) -> float:
    """
        Returns the input number multiplied by 3

        Args:
            num: The float number to be multiplied by 3
        Returns:
            Number that is multiplied by 3
    """
    return float(num) * 3

@tool
def get_current_date() -> date:
  """
    Returns the current date.
    Returns:
      The current date.
  """
  return date.today()

tools = [TavilySearch(max_results=100), triple, get_current_date]

llm = ChatOllama(model="qwen3.5:9b", temperature=0.3).bind_tools(tools)

In [ ]:
# nodes.py

from dotenv import load_dotenv
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode

load_dotenv()

SYSTEM_MESSAGE="""
You are a helpful assistant that can use the following tools to answer questions.
"""

# First node
def run_agent_reasoning(state: MessagesState) -> MessagesState:
    """
        Run the agent reasoning node.
    """
    response = llm.invoke([
        {
            "role": "system",
            "content": SYSTEM_MESSAGE,
        },
        *state["messages"]
    ])
    # below returned response will be appended to the list of messages state by LangGraph
    return { "messages": [response]}

# Second node
tool_node = ToolNode(tools=tools)

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph  import MessagesState, StateGraph, END

load_dotenv()

AGENT_REASON="agent_reason"
ACT="act"
LAST=-1

flow = StateGraph(MessagesState)

flow.add_node(AGENT_REASON, run_agent_reasoning)
flow.set_entry_point(AGENT_REASON)
flow.add_node(ACT, tool_node)


def should_continue(states: MessagesState) -> str:
    if not states["messages"][LAST].tool_calls:
        return END
    return ACT

flow.add_conditional_edges(AGENT_REASON, should_continue, {
    END: END,
    ACT: ACT
})

flow.add_edge(ACT, AGENT_REASON)

app = flow.compile()
app.get_graph().draw_mermaid_png(output_file_path="flow.png")


def main() -> None:
    print("Hello from langchain-course!")
    res = app.invoke({ "messages": [
        {
            "role": "user",
            "content": "Find and return me any publicly available list of 2 contacts (with email and phone number) of people from Learning and Development department from any Malaysian Government agencies."
        }
    ]})

    print(res["messages"][LAST].content)


if __name__ == "__main__":
    main()

Hello from langchain-course!
# Malaysian Government Agencies Overview

Based on the search results, here's information about key Malaysian government agencies and their functions:

## Investment & Economic Development

| Agency | Function |
|--------|----------|
| **MIDA** (Malaysian Investment Development Authority) | Promotes and develops investments under MITI. Coordinates investments in manufacturing and services sectors. Has 12 state offices and 21 overseas offices. |
| **Ministry of Economy** | Develops national economic policies and coordinates development programs. |
| **Development Division** | Prepares post-implementation reports and evaluates department/agency program outcomes. |

## Finance & Banking

| Agency | Function |
|--------|----------|
| **Ministry of Finance (MOF)** | Manages government finances, treasury, and fiscal policy. |
| **Central Bank of Malaysia** | Monetary authority and financial regulation. |
| **Bank Simpanan Nasional** | National savings bank. |
| *